In [51]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

cancer=load_breast_cancer()
scaler=StandardScaler()
X=cancer.data
Y=cancer.target
print(X.shape)
print(Y.shape)
print("Hello")
x_train,x_test,y_train,y_test=train_test_split(X,Y, test_size=0.25, random_state=42, stratify=Y)
model=KNeighborsClassifier(n_neighbors=10)
xtrain_scaled=scaler.fit_transform(x_train)
xtest_scaled=scaler.transform(x_test)
model.fit(xtrain_scaled,y_train)
y_pred=model.predict(xtest_scaled)
print(f"Accuracy {accuracy_score(y_test,y_pred)*100:.2f}")
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))




(569, 30)
(569,)
Hello
Accuracy 96.50
Detailed Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.92      0.95        53
      benign       0.96      0.99      0.97        90

    accuracy                           0.97       143
   macro avg       0.97      0.96      0.96       143
weighted avg       0.97      0.97      0.96       143



In [47]:

from sklearn.datasets import load_wine

wine = load_wine()
print(wine.feature_names)   # list of features
print(wine.data.shape)      # (178, 13)
print(wine.target)


['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
[[1.423e+01 1.710e+00 2.430e+00 ... 1.040e+00 3.920e+00 1.065e+03]
 [1.320e+01 1.780e+00 2.140e+00 ... 1.050e+00 3.400e+00 1.050e+03]
 [1.316e+01 2.360e+00 2.670e+00 ... 1.030e+00 3.170e+00 1.185e+03]
 ...
 [1.327e+01 4.280e+00 2.260e+00 ... 5.900e-01 1.560e+00 8.350e+02]
 [1.317e+01 2.590e+00 2.370e+00 ... 6.000e-01 1.620e+00 8.400e+02]
 [1.413e+01 4.100e+00 2.740e+00 ... 6.100e-01 1.600e+00 5.600e+02]]
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2
 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [52]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

def evaluate_model(X_train, X_test, y_train, y_test, name):
    clf = DecisionTreeClassifier(random_state=42, max_depth=5)
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    print(f"{name} Accuracy: {accuracy_score(y_test, pred) * 100:.2f}%")

# 1. Load and prepare the data
cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = cancer.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Scale the data (Crucial for PCA)
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("--- Baseline ---")
evaluate_model(X_train_scaled, X_test_scaled, y_train, y_test, "Baseline (All 30 Features)")


# ==========================================
# PATH A: Feature Selection (SelectKBest)
# ==========================================
print("\n--- Feature Selection (Keeping Top 5 Features) ---")

# Initialize SelectKBest to keep the 5 best features using ANOVA F-value (f_classif)
selector = SelectKBest(score_func=f_classif, k=5).set_output(transform="pandas")

# Fit on training data and transform both train/test
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)

# Print the names of the 5 winning features
selected_columns = X_train_selected.columns.tolist()
print(f"Kept Features: {selected_columns}")

evaluate_model(X_train_selected, X_test_selected, y_train, y_test, "Feature Selection")


# ==========================================
# PATH B: Dimensionality Reduction (PCA)
# ==========================================
print("\n--- Dimensionality Reduction (PCA to 5 Components) ---")

# Initialize PCA to condense 30 features into 5 components
pca = PCA(n_components=5).set_output(transform="pandas")

# Fit on training data and transform both train/test
# Note: PCA doesn't look at y_train. It only looks at the variance in X.
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Print how much original information these 5 components retained
variance_retained = pca.explained_variance_ratio_.sum() * 100
print(f"Variance (Information) Retained: {variance_retained:.2f}%")
print(f"New Feature Names: {X_train_pca.columns.tolist()}")

evaluate_model(X_train_pca, X_test_pca, y_train, y_test, "PCA")

--- Baseline ---
Baseline (All 30 Features) Accuracy: 93.71%

--- Feature Selection (Keeping Top 5 Features) ---
Kept Features: ['mean perimeter', 'mean concave points', 'worst radius', 'worst perimeter', 'worst concave points']
Feature Selection Accuracy: 92.31%

--- Dimensionality Reduction (PCA to 5 Components) ---
Variance (Information) Retained: 85.19%
New Feature Names: ['pca0', 'pca1', 'pca2', 'pca3', 'pca4']
PCA Accuracy: 92.31%
